# TESS: understand a task package, configure an experiment

Predict a star's near-core rotation frequency from its light curve. This notebook explains and edits configuration. The terminal launch script reads your saved experiment and starts the run.

**Before opening:** follow `tutorials/paper/README.md` in the exp repository. Start Jupyter from a terminal with the path variables and provider keys configured. Default Run All is read-only: no provider requests, training, downloads, or file writes.

## Two layers, two responsibilities

| Layer | Owns | Example edit |
|---|---|---|
| Task package | Scientific problem, data identities/splits, tensor contract, metric, validity rules | Clarify the prediction target in a copied task description |
| Experiment | Selects a task composition and workflow/treatment; sets model routing, rounds, budgets, hardware and output paths | Select the copied task; increase epochs from 1 to 2 |

The notebook teaches both layers and can save external files when you opt in. The **launch script** validates the saved experiment, resolves its task package, checks credentials/data/GPU, constructs the framework command and executes it. You can launch without Jupyter installed in the running process or leave this notebook closed.

Changing an experiment budget does not change the task metric or split. Changing the task description changes its fingerprint; reusing the old experiment name does not make it the original scientific treatment.


## Where your files live

Initialize a new external project using the README before opening this copied notebook. Neither repository is your editable project. `PROJECT` below is the user folder; `experiment.workspace` is a **single run's output folder** inside `PROJECT/runs/`. Keep these two paths different.

| File in your project | You edit it for | Read by |
|---|---|---|
| `tasks/tess/` | Scientific task declarations and implementation | Composition resolver |
| `experiments/tess-experiment.json` | Selected task, routing, budgets and output path | Launch script |
| `llm/agents.json` | Provider/model settings, **no keys** | Active agents |
| `advice/human_advice.example.json` | Learn the human-advice JSON format | **Inactive** in this NoPrior tutorial |
| `scripts/run-tess.sh` | Select the experiment file to launch | Your terminal |
| `notebooks/01_tess_tutorial.ipynb` | Explanation and editing exercises | Jupyter |
| `data/raw-tess/`, `data/tess-data/` | Downloaded source and staged run input | Preparation tool / runtime |
| `runs/<name>/`, `runs/<name>.tutorial.json` | Results and launch receipt | Written by script/runtime; read here |

Keep API keys in an external secret store/environment, outside the project and both repositories. Never put them in `llm/agents.json`. The source `.venv` is installation infrastructure; do not edit tracked source, task or experiment files.


In [ ]:
import csv
import json
import os
import shlex
import sys
from pathlib import Path

PROJECT = Path(os.environ["TUTORIAL_HOME"]).resolve()
binding = json.loads((PROJECT / "project.json").read_text())
EXP = Path(binding["exp_checkout"])
INFRA = Path(binding["infra_checkout"])
assert Path(sys.prefix).resolve() == (EXP / ".venv").resolve(), (
    "Select the exp tutorial kernel"
)
assert not PROJECT.is_relative_to(EXP) and not EXP.is_relative_to(PROJECT)
assert not PROJECT.is_relative_to(INFRA) and not INFRA.is_relative_to(PROJECT)
sys.dont_write_bytecode = True
# Import the installed checkout's tools. All edited files use absolute project paths.
os.chdir(EXP)
from tutorials.paper.project import write_launcher
from tutorials.paper.runner import TutorialExperiment, credential_status

TASK = PROJECT / "tasks/tess"
DATA = PROJECT / "data/tess-data"
experiment_path = PROJECT / "experiments/tess-experiment.json"
base_experiment = TutorialExperiment.model_validate_json(experiment_path.read_text())
print("User project:", PROJECT)
print("Task copy:", TASK)
print("Experiment:", experiment_path)
print("Run output (created by script only):", base_experiment.workspace)

## 1. Follow the files from task to run

The **composition** points to the data adapter, scientific declarations, metric and Health checks. The **workflow** chooses iterations, rounds and budgets. The **information treatment** decides which agent modules receive information. Changing one does not implicitly change the others.

| File | Owns |
|---|---|
| `compositions/rotation_regression.yaml` | Links the task's declarations and plugins |
| `declared/task_config.yaml` | Task description and model input/output contract |
| `data/manifests/rotation_identity.csv` | Exact train/validation row identities and labels |
| `runtime/tess_data_path.py` | Normalization, scope selection, tensor materialization |
| `declared/metric_r2.json` | Primary ranking metric; larger R² is better |
| `declared/task_health.yaml` | Checks on model outputs |
| `PROJECT/experiments/*.json` | Your runtime budget overrides and selected task |
| `PROJECT/llm/agents.json` | Your provider/model routing |

The model sees float32 `[B, 1, 1024]` input and returns `[B, 1]` rotation frequency in cycles/day. `B` is batch size. The input length is a task contract, not a free demo knob.

Task-relative paths above are under your `PROJECT/tasks/tess/` copy. The installed workflow supplies fixed defaults; edit your external experiment overrides, not the repository workflow.

In [ ]:
import yaml

composition = yaml.safe_load(
    (TASK / "compositions/rotation_regression.yaml").read_text()
)
print("YOUR task composition:")
print(yaml.safe_dump(composition, sort_keys=False))
print("YOUR experiment:")
print(base_experiment.model_dump_json(indent=2))

## 2. Understand which data the model may use

One row is a star observed in one sector. The released split has **3,338 training** and **442 validation** curves; the held-out **403 test** curves are not part of the run. Train/validation feedback does not establish independent test performance.

The next cell reads committed identities, not the external raw dataset. The task fixes preprocessing: truncate long curves, normalize observed values, then pad short curves with their last normalized value. Training targets are available for supervised learning; validation remains the scoring population.

In [ ]:
with (TASK / "data/manifests/rotation_identity.csv").open() as stream:
    rows = list(csv.DictReader(stream))
counts = {
    split: sum(row["split"] == split for row in rows) for split in ("train", "val")
}
assert counts == {"train": 3338, "val": 442}
print(counts)
print("Identity columns:", list(rows[0]))

### Download and stage from a terminal

This explicit command downloads **only** the pinned train/validation Parquet files, checks SHA-256, and calls the existing staging tool. It writes external data, but makes no LLM request. The destinations must be new. The source receipt stays in `raw-tess`; only the two NPZ archives enter `tess-data`.

In [ ]:
prepare_command = [
    str(EXP / ".venv/bin/python"),
    "-B",
    "-m",
    "tutorials.paper.prepare_tess",
    "--raw-dir",
    str(PROJECT / "data/raw-tess"),
    "--data-dir",
    str(DATA),
]
print("Run from the exp checkout in a terminal (output stays in YOUR project):")
print(shlex.join(prepare_command))

### Inspect one actual model input (after staging)

The task implementation below is the normalization authority. The plot is illustrative; it does not choose a training subset or change the data.

In [ ]:
import runpy

import numpy as np

# Inspect the same user-owned implementation selected by this task copy.
normalize_curve = runpy.run_path(str(TASK / "runtime/tess_data_path.py"))[
    "normalize_curve"
]

if (DATA / "tess_rotation_train.npz").is_file():
    import matplotlib.pyplot as plt

    with np.load(DATA / "tess_rotation_train.npz", allow_pickle=False) as archive:
        key = archive.files[0]
        raw = archive[key]
    prepared = normalize_curve(raw, 1024)
    print("Raw length:", len(raw), "model input:", prepared[None, None, :].shape)
    plt.plot(prepared)
    plt.xlabel("Time-sample index")
    plt.ylabel("Normalized flux")
    plt.show()
else:
    print("Data not staged. Run the printed preparation command, then rerun this cell.")

## 3. Modify your task copy

`TASK` points to `PROJECT/tasks/tess`, created during initialization. It never points at the repository's original task. You may edit this copy directly. To preserve it and make a second variant, enable the exercise below once; it copies the whole package into `tasks/tess-variant` and clarifies the target unit.

The next experiment explicitly selects `selected_composition`. Changing the task copy without selecting it in an experiment does not change another run. Scientific shape/split/metric changes require matching runtime changes and qualification. The current TESS preparation and population checks still target the released 3,338/442 population; this exercise changes description only.


In [ ]:
selected_composition = TASK / "compositions/rotation_regression.yaml"
COPY_TASK = False
if COPY_TASK:
    import shutil

    copy_root = PROJECT / "tasks/tess-variant"
    shutil.copytree(
        TASK, copy_root, ignore=shutil.ignore_patterns("__pycache__", "*.pyc")
    )
    config_file = copy_root / "declared/task_config.yaml"
    declaration = yaml.safe_load(config_file.read_text())
    declaration["task_description"] += (
        "\nReturn rotation frequency in cycles per day.\n"
    )
    config_file.write_text(yaml.safe_dump(declaration, sort_keys=False))
    selected_composition = copy_root / "compositions/rotation_regression.yaml"
    print("Selected task composition:", selected_composition)
else:
    print("Using your existing task copy; enable COPY_TASK to create a second variant.")

## 4. Modify your experiment

The initialized experiment already references your task and `llm/agents.json` by absolute path. Change budget values below, select the task variant from the previous cell, and choose a **new** run directory. For example, `epochs=2` changes training allowance without changing the task metric or split.

The script reads the saved JSON, not this notebook's variables. Save a new experiment and generate its script in section 6. One iteration provides Trial and Formal opportunities; failure can prevent a Formal result. The 2/5-minute attempt budgets exclude LLM waiting and are not a total cost/time cap. This entrypoint fixes the TESS workflow and NoPrior treatment.


In [ ]:
values = base_experiment.model_dump(mode="json")
values.update(
    composition=str(selected_composition),
    llm_config=str(PROJECT / "llm/agents.json"),
    workspace=str(PROJECT / "runs/tess-edited-001"),
    run_name="tess_edited_001",
    epochs=2,
)
experiment = TutorialExperiment.model_validate(values)
print(experiment.model_dump_json(indent=2))

## 5. Check the information treatment and API requirements

TESS NoPrior disables **advice, Data Analysis and literature review**. That differs from the TIDMAD NoPrior treatment. No Semantic Scholar key is needed here.

The shipped active agents use `OPENAI_API_KEY`. In the **launching terminal**, use:

```bash
read -r -s -p "OpenAI API key: " OPENAI_API_KEY
echo
export OPENAI_API_KEY
```

Or load a trusted mode-600 external environment file. Do not put keys in notebook cells, JSON settings, output, or Git. Notebook and terminal environments can differ. Presence is not a provider-access test; this notebook makes no request.

**Warning:** configure keys before starting Jupyter from that terminal. If Jupyter was already running, stop its server and restart it from the configured terminal, then restart the kernel. A kernel restart alone cannot update the parent server environment. Configure any separate launch terminal too. Checks show variable names and presence only; they never display values or make an API request. A file existing on disk does not mean its keys were exported.

### Human advice is a separate experiment input

`advice/human_advice.example.json` is a local copy of the existing JSON example, not a prompt text file or API-key file. This NoPrior entrypoint does **not** read it: `advice_file` must remain `null`. Merely editing or renaming it does not enable advice. A non-null value is refused. An advice-enabled experiment needs a separate information treatment with a new identity, the advice artifact's SHA-256, and explicit routing through the framework's advice loader. This tutorial does not silently switch to that treatment.


In [ ]:
routing_path = experiment.llm_config
assert routing_path is not None
routing = json.loads(routing_path.read_text())
print("YOUR routing file:", routing_path)
print("Tuner routing:", routing["tune"])
key_status = credential_status(routing_path)
for name, present in key_status.items():
    print(f"{name}: {'configured' if present else 'MISSING'}")
if not all(key_status.values()):
    print(
        "WARNING: export missing keys before starting Jupyter and the launch script. "
        "Never paste secret values into notebook cells. Offline reading can continue."
    )
print("Advice example is inactive; advice_file:", experiment.advice_file)

## 6. Save a new experiment and generate your script

Set `WRITE_CONFIG = True` once. This writes a new experiment and a new script **inside your project**, refusing to overwrite existing files. Edit `llm/agents.json` locally to change model routing; if you want to preserve it, copy it to another local filename and set `llm_config` to that file before saving.

The generated shell script explicitly selects the saved experiment. It calls the installed runner using the exp checkout's interpreter; it does not copy or modify framework source. You can open your generated script, change its `EXPERIMENT` path to another saved local experiment, and launch from any terminal directory. Keep secrets out of it.


In [ ]:
WRITE_CONFIG = False
config_path = PROJECT / "experiments/tess-edited.json"
script_path = PROJECT / "scripts/run-tess-edited.sh"
if WRITE_CONFIG:
    assert not config_path.exists() and not script_path.exists(), (
        "Choose fresh filenames"
    )
    with config_path.open("x") as stream:
        stream.write(experiment.model_dump_json(indent=2))
    write_launcher(script_path, config_path, INFRA)
    print("Saved:", config_path)
    print("Generated:", script_path)
else:
    print("Read-only. Enable WRITE_CONFIG to create your experiment and launch script.")

## 7. Preview and launch from the terminal

First run the preview; inspect its source revisions, composition fingerprint, required key names and argv. Then add `--launch`. The notebook only prints commands. The script reads the saved experiment from disk, not the live Python object in this notebook. After changing a cell, save a new experiment file before launching it.

The preview requires clean exp/infra source and the pinned environments. It resolves trusted task code but calls no provider and runs no training. Launch additionally verifies both data populations, checks the GPU and executes a CUDA allocation. The run stays under your ordinary user; do not use sudo.

In [ ]:
selected_key_status = credential_status(experiment.llm_config)
print("Selected routing key presence:", selected_key_status)
if not all(selected_key_status.values()):
    print("WARNING: configure required environment variables before launch.")
run_command = ["bash", str(script_path)]
print("After saving section 6, run your generated script:")
print("PREVIEW:", shlex.join(run_command))
print("LAUNCH (paid API calls and training):", shlex.join(run_command + ["--launch"]))
print("Results:", experiment.workspace)

## 8. Inspect results without assuming success

The wrapper writes a sibling `<workspace>.tutorial.json` receipt. The framework writes node records and per-iteration manifests under the workspace. A Trial score is not a Formal score; an absent result is not zero. Inspect status, scope, primary metric, Health and provenance before comparing candidates.

The next cell lists likely records without dumping model code, full prompts or credentials. Open a selected record deliberately. A demo checks the path through the system, not whether it beats a published baseline.

In our RTX 5090 smoke run, the first tiny Trial was refused because 2 training batches and 1 validation batch could not establish runtime calibration. The workflow retried with full scopes and batch size 4 and obtained a finite Trial score. One epoch still produced nearly constant predictions. Read failed and scored attempts together; a score alone does not establish task Health or paper-quality performance. This observed recovery is not guaranteed for another proposal.

The run completed both rounds, but its Formal R² was about -0.0625 and the observational prediction-dispersion check failed. An observational check can fail while the run completes; `blocking` mode does not turn every check into a blocking rule. The result authority is `diagnostic`, so it cannot establish paper reproduction. A failure record with no persisted logical round is shown as `Unspecified`; the reader does not infer Formal from a default false `is_trial` field.

In [ ]:
receipt_path = experiment.workspace.with_name(
    experiment.workspace.name + ".tutorial.json"
)
if receipt_path.is_file():
    receipt = json.loads(receipt_path.read_text())
    print(
        {
            key: receipt[key]
            for key in (
                "kind",
                "scientific_reproduction",
                "infra_revision",
                "exp_revision",
                "composition_fingerprint",
            )
        }
    )
if experiment.workspace.is_dir():
    records = sorted(experiment.workspace.rglob("run_output_*.json"))
    manifests = sorted(experiment.workspace.rglob("*manifest*.json"))
    print(
        "Tuner records:",
        [str(p.relative_to(experiment.workspace)) for p in records[:20]],
    )
    print(
        "Manifests:", [str(p.relative_to(experiment.workspace)) for p in manifests[:20]]
    )
else:
    print("No run workspace yet. This is expected before launch.")

In [ ]:
# Optional: choose one run_output_*.json from the list above.
SELECTED_RECORD = None  # e.g. experiment.workspace / ".../run_output_....json"
if SELECTED_RECORD is not None:
    import pandas as pd
    from agent.schemas.hyperparam_tuning import HyperparamTuningOutput

    output = HyperparamTuningOutput.model_validate_json(
        Path(SELECTED_RECORD).read_text()
    )
    display(
        pd.DataFrame(
            [
                {
                    "round": record.logical_round,
                    "role": ("Trial" if record.is_trial else "Formal")
                    if record.logical_round is not None
                    else "Unspecified",
                    "status": record.status,
                    "metric": record.metric_result.metric_id
                    if record.metric_result
                    else None,
                    "score": record.metric_result.scalar
                    if record.metric_result
                    else None,
                    "health_checks": [
                        {
                            "name": gate.gate_name,
                            "passed": gate.check_passed,
                            "role": gate.gate_role,
                            "action": gate.resolved_action,
                        }
                        for gate in record.health_gate_results
                    ],
                    "authority": record.scientific_authority,
                }
                for record in output.all_records
            ]
        )
    )
else:
    print(
        "Select a saved tuner record after a run; no scores are invented before then."
    )

## 9. Where the paper run and sandbox differ

For the six-hour RTX 5090 experiment, use the original `experiments/phyts_tess/main_fixed_workflow/launch.sh` with `--arm no-prior`, as shown in the tutorial README. Its immutable deadline and resume behavior are not part of this short demo adapter. It does not read demo settings JSON. The current source pin includes fixes after historical runs; exact archived replay requires the run's own pinned source and checkpoints.

Fixed workflow does not expose an external coding-agent shell. It still executes generated model code in subprocesses. This is **not an OS security sandbox**: readable files and validation targets are not made private from arbitrary code under the same user. Task declarations, process resource controls, filesystem permissions and private-evaluator isolation are different boundaries.

Changing NVIDIA GPU is a new hardware treatment; the same time budget can explore fewer or more models. AMD/Intel GPU support is not provided. Preserve the data split, record changed settings, and compare scores only over the same population.